**Materia:** Tecnologías Emergentes — Primavera 2026  
**Profesor:** Mtro. Rafael Pérez Aguirre

# 📚 RAG (Retrieval-Augmented Generation)

**RAG** es una técnica que combina la generación de texto de un LLM con la **recuperación de información** de una base de conocimiento externa. En lugar de depender únicamente del conocimiento interno del modelo (que puede estar desactualizado o ser incompleto), el agente primero **busca** documentos relevantes y luego los usa como contexto para generar una respuesta fundamentada.

Esto es clave para construir asistentes que respondan con información precisa, actualizada y específica de un dominio — como manuales internos, documentación técnica o bases de conocimiento empresariales.

📖 [Documentación oficial: RAG](https://docs.langchain.com/oss/python/langchain/rag)

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d" width="700">

## Configuración

Cargar y/o verificar las variables de entorno necesarias

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

variables_requeridas = ["GEMINI_API_KEY"]
for var in variables_requeridas:
    if os.getenv(var):
        print(f"✅ {var} cargada correctamente")
    else:
        print(f"❌ {var} no encontrada")

✅ GEMINI_API_KEY cargada correctamente


## Preparar los documentos

El primer paso de RAG es tener una **base de conocimiento**. En este caso, simularemos documentos sobre el reglamento de una universidad ficticia. En producción, estos documentos podrían venir de PDFs, páginas web, bases de datos, etc.

Usamos `Document` de LangChain para representar cada fragmento de texto con su metadata.

In [2]:
from langchain_core.documents import Document

documentos = [
    Document(
        page_content="El horario de clases es de lunes a viernes de 7:00 a 22:00 horas. "
        "Los sábados se imparten clases de 8:00 a 14:00. Los domingos la universidad permanece cerrada.",
        metadata={"fuente": "reglamento", "seccion": "horarios"},
    ),
    Document(
        page_content="Para inscribirse a materias, los alumnos deben acceder al portal académico "
        "durante el periodo de inscripciones. El calendario se publica en la página principal. "
        "Cada alumno puede inscribir un máximo de 10 materias por semestre.",
        metadata={"fuente": "reglamento", "seccion": "inscripciones"},
    ),
    Document(
        page_content="La calificación mínima aprobatoria es 60 sobre 100. "
        "Los alumnos tienen derecho a dos exámenes ordinarios y un extraordinario por materia. "
        "El extraordinario tiene un costo adicional de $10000 MXN.",
        metadata={"fuente": "reglamento", "seccion": "evaluaciones"},
    ),
    Document(
        page_content="La biblioteca está abierta de lunes a viernes de 7:00 a 21:00 y sábados de 8:00 a 13:00. "
        "Los préstamos de libros son por un máximo de 7 días con posibilidad de renovación. "
        "Se permiten hasta 3 libros simultáneos por alumno.",
        metadata={"fuente": "servicios", "seccion": "biblioteca"},
    ),
    Document(
        page_content="El servicio de becas cubre del 25% al 100% de la colegiatura. "
        "Los requisitos son: promedio mínimo de 85, no tener materias reprobadas "
        "y realizar 60 horas de servicio comunitario por semestre.",
        metadata={"fuente": "servicios", "seccion": "becas"},
    ),
    Document(
        page_content="El estacionamiento tiene un costo de $300 MXN mensuales para alumnos "
        "y es gratuito para profesores. El horario del estacionamiento es de 6:30 a 22:30 "
        "de lunes a viernes y de 7:30 a 14:30 los sábados.",
        metadata={"fuente": "servicios", "seccion": "estacionamiento"},
    ),
    Document(
        page_content="Para titulación, el alumno debe haber cubierto el 100% de créditos, "
        "completar el servicio social (480 horas), aprobar el examen TOEFL con mínimo 500 puntos "
        "y presentar un proyecto integrador o tesis.",
        metadata={"fuente": "reglamento", "seccion": "titulacion"},
    ),
    Document(
        page_content="La cafetería ofrece desayunos de 7:00 a 10:00 y comidas de 12:00 a 16:00. "
        "Los precios van de $35 a $75 MXN por platillo. "
        "Existe un menú especial para alumnos con dietas vegetarianas y veganas.",
        metadata={"fuente": "servicios", "seccion": "cafeteria"},
    ),
]

print(f"- {len(documentos)} documentos creados")
print(f"\nEjemplo de documento:")
print(f"  Contenido: {documentos[0].page_content[:80]}...")
print(f"  Metadata:  {documentos[0].metadata}")

- 8 documentos creados

Ejemplo de documento:
  Contenido: El horario de clases es de lunes a viernes de 7:00 a 22:00 horas. Los sábados se...
  Metadata:  {'fuente': 'reglamento', 'seccion': 'horarios'}


## Crear el Vector Store

Para que el agente pueda **buscar** documentos relevantes, necesitamos convertir el texto en **embeddings** (vectores numéricos) y almacenarlos en un **vector store**.

Usamos:
- `OpenAIEmbeddings` para generar los vectores
- `InMemoryVectorStore` como almacén simple (ideal para demos)

Cuando hacemos una búsqueda, el vector store compara la similitud entre el embedding de la pregunta y los de los documentos.

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import os

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY"),
)

vector_store = InMemoryVectorStore.from_documents(
    documents=documentos,
    embedding=embeddings,
)

print("- Vector store creado con", len(documentos), "documentos")


- Vector store creado con 8 documentos


In [9]:
# Probar una búsqueda por similitud directamente
resultados = vector_store.similarity_search("¿Cuánto cuesta el estacionamiento?", k=2)

print("Resultados de búsqueda:\n")
for i, doc in enumerate(resultados, 1):
    print(f"── Resultado {i} ──")
    print(f"\tSección: {doc.metadata['seccion']}")
    print(f"\tFuente:  {doc.metadata['fuente']}")
    print(f"\tContenido:\n\t{doc.page_content}")

Resultados de búsqueda:

── Resultado 1 ──
	Sección: estacionamiento
	Fuente:  servicios
	Contenido:
	El estacionamiento tiene un costo de $300 MXN mensuales para alumnos y es gratuito para profesores. El horario del estacionamiento es de 6:30 a 22:30 de lunes a viernes y de 7:30 a 14:30 los sábados.
── Resultado 2 ──
	Sección: cafeteria
	Fuente:  servicios
	Contenido:
	La cafetería ofrece desayunos de 7:00 a 10:00 y comidas de 12:00 a 16:00. Los precios van de $35 a $75 MXN por platillo. Existe un menú especial para alumnos con dietas vegetarianas y veganas.


Observa cómo la búsqueda por similitud devuelve los documentos más **semánticamente cercanos** a la pregunta, aunque no contengan las mismas palabras exactas.

También podemos usar `similarity_search_with_score` para obtener el **score de similitud** de cada resultado. Esto es útil para evaluar qué tan relevante es cada documento.

In [10]:
# Búsqueda con score de similitud
resultados_con_score = vector_store.similarity_search_with_score("¿Cuánto cuesta el estacionamiento?", k=2)

print("Resultados con score de similitud:\n")
for i, (doc, score) in enumerate(resultados_con_score, 1):
    print(f"── Resultado {i} ──")
    print(f"\tSección: {doc.metadata['seccion']}")
    print(f"\tFuente:  {doc.metadata['fuente']}")
    print(f"\tScore:   {score:.4f}")
    print(f"\tContenido:\n\t{doc.page_content}\n")

Resultados con score de similitud:

── Resultado 1 ──
	Sección: estacionamiento
	Fuente:  servicios
	Score:   0.7435
	Contenido:
	El estacionamiento tiene un costo de $300 MXN mensuales para alumnos y es gratuito para profesores. El horario del estacionamiento es de 6:30 a 22:30 de lunes a viernes y de 7:30 a 14:30 los sábados.

── Resultado 2 ──
	Sección: cafeteria
	Fuente:  servicios
	Score:   0.6091
	Contenido:
	La cafetería ofrece desayunos de 7:00 a 10:00 y comidas de 12:00 a 16:00. Los precios van de $35 a $75 MXN por platillo. Existe un menú especial para alumnos con dietas vegetarianas y veganas.



## Crear el Agente RAG

Ahora combinamos todo: creamos un **tool de búsqueda** que el agente puede invocar para consultar la base de conocimiento, y luego construimos el agente.

El flujo RAG completo es:
1. El usuario hace una pregunta
2. El agente decide buscar información relevante (usa el tool)
3. El tool devuelve los documentos más similares
4. El agente genera una respuesta fundamentada en esos documentos

In [11]:
from langchain_core.tools import tool


@tool
def buscar_reglamento(consulta: str) -> str:
    """Busca información en el reglamento y servicios de la universidad.
    Usa esta herramienta cuando el usuario pregunte sobre horarios, inscripciones,
    calificaciones, becas, biblioteca, cafetería, estacionamiento o titulación.

    Args:
        consulta (str): La pregunta o tema a buscar en la base de conocimiento.

    Returns:
        str: Los documentos más relevantes encontrados, separados por sección.
    """
    resultados = vector_store.similarity_search(consulta, k=3)
    textos = []
    for doc in resultados:
        textos.append(f"[{doc.metadata['seccion']}]: {doc.page_content}")
    return "\n\n".join(textos)


print(f"- Tool creado: {buscar_reglamento.name}")
print(f"   Descripción: {buscar_reglamento.description[:80]}...")

- Tool creado: buscar_reglamento
   Descripción: Busca información en el reglamento y servicios de la universidad.
Usa esta herra...


In [15]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

agente_rag = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
    tools=[buscar_reglamento],
    system_prompt=(
        "Eres un asistente virtual de la universidad. "
        "Responde preguntas de los alumnos usando ÚNICAMENTE la información "
        "del reglamento y servicios disponibles a través del tool de búsqueda. "
        "Si no encuentras la información, di que no tienes esa información disponible. "
        "Responde siempre en español de forma clara y amable."
    ),
)

print("- Agente RAG creado")


- Agente RAG creado


## Consultas al agente

Probemos el agente con distintas preguntas. Observa cómo:
- El agente **decide** cuándo necesita buscar información
- Usa los documentos recuperados para **fundamentar** su respuesta
- No inventa información que no esté en los documentos

Puedes ver los traces completos en [LangSmith](https://smith.langchain.com).

In [13]:
# Pregunta directa sobre un tema específico
resultado = agente_rag.invoke(
    {"messages": [{"role": "user", "content": "¿Cuáles son los requisitos para obtener una beca?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

================================ Human Message =================================

¿Cuáles son los requisitos para obtener una beca?
================================== Ai Message ==================================

[]
Tool Calls:
  buscar_reglamento (3fede614-c96e-4fae-9b88-c55f4905ffc9)
 Call ID: 3fede614-c96e-4fae-9b88-c55f4905ffc9
  Args:
    consulta: requisitos beca
================================= Tool Message =================================
Name: buscar_reglamento

[becas]: El servicio de becas cubre del 25% al 100% de la colegiatura. Los requisitos son: promedio mínimo de 85, no tener materias reprobadas y realizar 60 horas de servicio comunitario por semestre.

[titulacion]: Para titulación, el alumno debe haber cubierto el 100% de créditos, completar el servicio social (480 horas), aprobar el examen TOEFL con mínimo 500 puntos y presentar un proyecto integrador o tesis.

[evaluaciones]: La calificación mínima aprobatoria es 60 sobre 100. Los alumnos tienen derecho a dos exá

In [16]:
# Pregunta que requiere buscar en una sección específica
resultado = agente_rag.invoke(
    {"messages": [{"role": "user", "content": "¿Hasta qué hora puedo estar en la biblioteca los sábados?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 18.410682735s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [ ]:
# Pregunta sobre titulación
resultado = agente_rag.invoke(
    {"messages": [{"role": "user", "content": "¿Qué necesito para titularme?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

In [ ]:
# Pregunta sobre algo que NO está en los documentos
resultado = agente_rag.invoke(
    {"messages": [{"role": "user", "content": "¿Cuál es el correo del departamento de sistemas?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

## RAG con documentos externos

En un caso más realista, los documentos provienen de archivos externos. LangChain ofrece **document loaders** para cargar texto desde PDFs, páginas web, archivos CSV, etc.

También podemos usar **text splitters** para dividir documentos largos en fragmentos más pequeños, lo que mejora la precisión de la búsqueda.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cargar una página web como ejemplo
loader = WebBaseLoader("https://es.wikipedia.org/wiki/Inteligencia_artificial")
documentos_web = loader.load()

print(f"- Documentos cargados: {len(documentos_web)}")
print(f"   Caracteres total: {len(documentos_web[0].page_content)}")

In [ ]:
# Dividir en fragmentos más pequeños
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

fragmentos = splitter.split_documents(documentos_web)

print(f"- Fragmentos generados: {len(fragmentos)}")
print(f"\nEjemplo de fragmento:")
print(fragmentos[0].page_content, "...")


In [ ]:
# Crear vector store con los fragmentos de la web
vector_store_web = InMemoryVectorStore.from_documents(
    documents=fragmentos,
    embedding=embeddings,
)


@tool
def buscar_wikipedia(consulta: str) -> str:
    """Busca información sobre inteligencia artificial en la base de conocimiento.

    Args:
        consulta (str): La pregunta o tema a buscar sobre inteligencia artificial.

    Returns:
        str: Los fragmentos de texto más relevantes encontrados en la base de conocimiento.
    """
    resultados = vector_store_web.similarity_search(consulta, k=3)
    textos = [doc.page_content for doc in resultados]
    return "\n\n".join(textos)


agente_wiki = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
    tools=[buscar_wikipedia],
    system_prompt=(
        "Eres un asistente experto en inteligencia artificial. "
        "Usa el tool de búsqueda para fundamentar tus respuestas. "
        "Responde en español."
    ),
)

print("- Agente con documentos web creado")


In [ ]:
resultado = agente_wiki.invoke(
    {"messages": [{"role": "user", "content": "¿Qué es la inteligencia artificial y cuáles son sus orígenes?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

In [ ]:
resultado = agente_wiki.invoke(
    {"messages": [{"role": "user", "content": "¿Dame los nombres de los persoajes más importantes de la IA?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

## Actividad

### Ejercicio 1: Agrega más documentos
Agrega al menos **3 documentos nuevos** a la base de conocimiento de la universidad (por ejemplo: deportes, laboratorios, intercambios) y hazle preguntas al agente.

In [ ]:
# Agrega tus documentos aquí — información nueva sobre servicios universitarios
nuevos_documentos = [
    Document(
        page_content="Las instalaciones deportivas incluyen cancha de fútbol, basquetbol, voleibol y tenis. "
        "El gimnasio está disponible de lunes a viernes de 6:00 a 22:00 y sábados de 8:00 a 14:00. "
        "El costo para alumnos es de $200 MXN mensuales.",
        metadata={"fuente": "servicios", "seccion": "deportes"},
    ),
    Document(
        page_content="Los laboratorios de cómputo están abiertos de lunes a viernes de 7:00 a 21:00. "
        "Cada laboratorio cuenta con 30 equipos con software especializado. "
        "Los docentes deben reservarlos con 48 horas de anticipación.",
        metadata={"fuente": "servicios", "seccion": "laboratorios"},
    ),
    Document(
        page_content="El programa de intercambio permite estudiar un semestre en universidades de EE.UU., Europa y Latinoamérica. "
        "Requisitos: promedio mínimo de 80, inglés nivel B2 y al menos 50% de créditos cursados. "
        "Las convocatorias abren en febrero y agosto de cada año.",
        metadata={"fuente": "servicios", "seccion": "intercambios"},
    ),
]

# Agregar al vector store existente
vector_store.add_documents(nuevos_documentos)

print(f"- {len(nuevos_documentos)} documentos agregados")

In [ ]:
# Pregunta sobre los datos nuevos
resultado = agente_rag.invoke(
    {"messages": [{"role": "user", "content": "¿Cuáles son los requisitos para ir a un intercambio estudiantil?"}]}
)

for msg in resultado["messages"]:
    msg.pretty_print()

### Ejercicio 2: RAG con tu propio tema
Crea un agente RAG sobre un tema de tu elección. Puedes cargar documentos desde una URL con `WebBaseLoader` o crear los documentos manualmente. El agente debe tener un tool de búsqueda y un system prompt apropiado.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent
import os

# Ejercicio 2: RAG sobre recetas de cocina mexicana
recetas = [
    Document(page_content="Los tacos al pastor se preparan con carne de cerdo marinada en achiote, chile chipotle y piña. Se ensarta en un trompo y se cocina lentamente. Se sirve en tortillas de maíz con cilantro, cebolla y salsa verde."),
    Document(page_content="El mole negro combina más de 20 ingredientes: chiles mulato, ancho y pasilla, chocolate amargo, plátano macho frito y especias. Se cocina por horas y se sirve sobre pollo o guajolote."),
    Document(page_content="Los chiles en nogada llevan chile poblano relleno de picadillo de carne, frutas y especias. Se bañan en nogada de nuez y se adornan con granada y perejil, representando los colores de la bandera mexicana."),
    Document(page_content="La sopa de lima es típica de Yucatán. Se hace con caldo de pollo, lima agria, tortilla frita, tomate, cebolla y chile. Se caracteriza por su sabor cítrico y refrescante."),
    Document(page_content="Los tamales de rajas con queso se hacen con masa de maíz preparada con manteca vegetal, rellenos con tiras de chile poblano asado y queso Oaxaca. Se envuelven en hojas de maíz y se cuecen al vapor durante 1 hora."),
]

embeddings_recetas = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY"),
)
vs_recetas = InMemoryVectorStore.from_documents(recetas, embedding=embeddings_recetas)

@tool
def buscar_receta(consulta: str) -> str:
    """Busca información sobre recetas de cocina mexicana tradicional.

    Args:
        consulta (str): El platillo o ingrediente a buscar.

    Returns:
        str: Información relevante sobre las recetas encontradas.
    """
    resultados = vs_recetas.similarity_search(consulta, k=2)
    return "\n\n".join([doc.page_content for doc in resultados])

agente_chef = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
    tools=[buscar_receta],
    system_prompt=(
        "Eres un chef experto en cocina mexicana tradicional. "
        "Usa el tool de búsqueda para fundamentar tus respuestas. "
        "Responde en español."
    ),
)

resultado_chef = agente_chef.invoke(
    {"messages": [{"role": "user", "content": "¿Cómo se preparan los chiles en nogada?"}]}
)
for msg in resultado_chef["messages"]:
    msg.pretty_print()
